# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [76]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [77]:
from langchain_community.document_loaders import PyPDFLoader

# Loading pdf 'The GenAI Divide: State of AI in Business 2025' documents object
loader = PyPDFLoader("../02_activities/documents/ai_report_2025.pdf")
documents = loader.load()

In [78]:

# Output is a collection of pages so joining them using the for loop below
document_text = ""
for page in documents:
    document_text += page.page_content + "\n"

# print(document_text)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [79]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os

# Setting up model configuration
MODEL = "gpt-4o-mini"
GATEWAY_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"
API_KEY = os.getenv('API_GATEWAY_KEY')

# Creating Pydantic model and defining the fields of the object
class DocumentAnalysis(BaseModel):
    author: str = Field(description="Author of the document")
    title: str = Field(description="Title of the document")
    relevance: str = Field(description="Why this is relevant for AI professionals")
    summary: str = Field(description="Summary in cricket commentator tone, max 1000 tokens")
    tone: str = Field(description="The tone used in the summary")
    input_tokens: int = Field(description="Number of input tokens")
    output_tokens: int = Field(description="Number of output tokens")

# Defining developer prompt/System prompt with instructions
SYSTEM_PROMPT = """You are an expert document analyst. Analyze the provided document and extract structured insights.
Your response must be valid JSON matching this exact structure:
{
    "author": "string",
    "title": "string",
    "relevance": "string (one paragraph max)",
    "summary": "string (max 1000 tokens)",
    "tone": "Cricket Commentator"
}"""

# User prompt template to analyze the document as if they were cricket commentator narrating a test match
USER_PROMPT_TEMPLATE = """Please analyze this document and write the summary as if you were a cricket commentator narrating a first class test match.

Document Content:
{document_text}"""
# Initialize OpenAI client with gateway.
def get_client() -> OpenAI:
 
    return OpenAI(
        base_url=GATEWAY_URL,
        api_key="any value",
        default_headers={"x-api-key": API_KEY}
    )
# Analysis of document with structured output done here by passing to the LLM client the document along with developer prompt and user prompt
def analyze_document(client: OpenAI, document_text: str) -> DocumentAnalysis:
    
    # Formating user prompt defined above with document
    user_prompt = USER_PROMPT_TEMPLATE.format(document_text=document_text)
    
    # Call OpenAI with structured output and passing system prompt and user prompt
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        response_format=DocumentAnalysis
    )
    
    # Extracting response into analysis object containging the parsed message, intput & output tokens
    analysis = response.choices[0].message.parsed
    analysis.input_tokens = response.usage.prompt_tokens
    analysis.output_tokens = response.usage.completion_tokens
    
    return analysis

# Displaying result of usage
client = get_client()
result = analyze_document(client, document_text)
print(result.model_dump_json(indent=2))

{
  "author": "MIT NANDA",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "This document is crucial for AI professionals as it dissects the current state of AI adoption in business, showcasing the stark realities of implementation and the critical learning gaps that impede the transition from pilot projects to scalable solutions. Understanding these challenges is fundamental for developing effective strategies and products in the rapidly evolving AI landscape.",
  "summary": "Ladies and gentlemen, as we step into the grand arena of artificial intelligence, we find ourselves amidst the GenAI Divide, where high hopes meet hard realities! Despite a hefty investment of $30 to $40 billion, a staggering 95% of organizations find themselves getting absolutely zero return on their AI ventures! Only a mere 5% of integrated pilots are hitting it big, raking in millions, while the rest are left scrambling for any measurable impact on profits. From the prestigious zone

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [80]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from deepeval import evaluate
from pydantic import BaseModel, Field
import os
from io import StringIO
import sys

# To suppress DeepEval's verbose output so stdout is not flooded with document text
os.environ['DEEPEVAL_VERBOSE'] = 'false'

# Setting up model configuration
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

model = GPTModel(
    model=MODEL,
    temperature=1,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# Defining output structure and evaluation metrics 
class EvaluationResults(BaseModel):
    summarization_score: float = Field(description="Summarization score")
    summarization_reason: str = Field(description="Summarization reason")
    coherence_score: float = Field(description="Coherence score")
    coherence_reason: str = Field(description="Coherence reason")
    tonality_score: float = Field(description="Tonality score")
    tonality_reason: str = Field(description="Tonality reason")
    safety_score: float = Field(description="Safety score")
    safety_reason: str = Field(description="Safety reason")

# Define metric definitions to pass to the client
summarization_metric = GEval(
    name="Summarization",
    evaluation_steps=[
        "Does the summary capture the main theme of the GenAI Divide and business implications?",
        "Are key statistics (95% adoption, $30-40B spending) accurately reflected?",
        "Does it explain why most AI initiatives fail to generate ROI?",
        "Is the role of partnerships and external vendors discussed?",
        "Does it identify that methodology (not model quality) causes the divide?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Are logical connections between paragraphs clear and well-structured?",
        "Does it follow a coherent narrative from problem to solution?",
        "Are technical terms explained or contextually clear?",
        "Is there logical flow between statistics, analysis, and recommendations?",
        "Do sentences connect smoothly without abrupt topic shifts?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Does it consistently use cricket commentator language throughout?",
        "Are cricket-specific metaphors or terminology appropriately applied?",
        "Is the formal, descriptive tone of cricket match commentary maintained?",
        "Does it employ characteristic pacing and rhythm of sports commentary?",
        "Are observations presented with analytical style of professional cricket commentator?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does it avoid unfounded or exaggerated claims about AI?",
        "Are conclusions supported by research and data?",
        "Does it avoid harmful stereotypes or biased statements?",
        "Are risks and limitations discussed responsibly?",
        "Does it avoid promoting unethical AI practices?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)



# Evaluating summary using DeepEval metrics, capture and suppress stdout
def evaluate_summary(document_content: str, summary: str) -> EvaluationResults:
    
    test_case = LLMTestCase(
        input=document_content,
        actual_output=summary
    )
    
    # Suppress verbose output
    old_stdout = sys.stdout
    sys.stdout = StringIO()
    
    try:
        results = evaluate(
            test_cases=[test_case],
            metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric]
        )
    finally:
        sys.stdout = old_stdout
    
    metrics_data = results.test_results[0].metrics_data
    
    return EvaluationResults(
        summarization_score=metrics_data[0].score,
        summarization_reason=metrics_data[0].reason,
        coherence_score=metrics_data[1].score,
        coherence_reason=metrics_data[1].reason,
        tonality_score=metrics_data[2].score,
        tonality_reason=metrics_data[2].reason,
        safety_score=metrics_data[3].score,
        safety_reason=metrics_data[3].reason
    )

# Printing evaluation results with summary
eval_results = evaluate_summary(document_text, result.summary)
print(eval_results.model_dump_json(indent=2))

✨ You're running DeepEval's latest Summarization [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

✓ Evaluation completed 🎉! (time taken: 6.0s | token cost: 0.0070076999999999995 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{
  "summarization_score": 0.7335227764675813,
  "summarization_reason": "The response effectively captures the main theme of the GenAI Divide and discusses business implications, highlighting the disparity between high investment and low ROI. It references the key statistic of 95% of organizations receiving no return and 5% of pilots succeeding, though it could have provided more analysis on partnership roles and methodology causing the divide. While it addresses some aspects of why AI initiatives fail, it lacks in-depth discussion on partnerships and the specific learning gap that defines the divide.",
  "coherence_score": 0.6566399834338191,
  "coherence_reason": "The response presents a clear narrative summarizing the main issues outlined in the input regarding the GenAI Divide, showcasing a coherent flow from problem identification to potential solutions. However, while there are logical connections, some technical terms such as 'contextual learning' and 'brittle workflows' are no

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [81]:
# Analyze evaluation results and create improvement prompt
def create_improvement_prompt(
    document_content: str,
    original_summary: str,
    evaluation_results: EvaluationResults,
    original_title: str
) -> str:
    """Create prompt to improve summary based on evaluation feedback."""
    
    improvement_prompt = f"""You are refining a document summary based on evaluation feedback.

ORIGINAL SUMMARY:
{original_summary}

EVALUATION FEEDBACK:
- Summarization: {evaluation_results.summarization_reason}
- Coherence: {evaluation_results.coherence_reason}
- Tonality: {evaluation_results.tonality_reason}
- Safety: {evaluation_results.safety_reason}

DOCUMENT CONTENT:
{document_content}

Please create an IMPROVED summary that addresses these issues:
1. Explicitly discuss the role of external vendors and partnerships
2. Make clear connections between methodology and the GenAI Divide
3. Improve narrative flow from problem to solution
4. Enhance cricket commentator tone with more specific cricket terminology
5. Add more detail on risks and ethical implications

Write the summary in cricket commentator tone as if narrating a first class test match. Keep it under 1000 tokens."""
    
    return improvement_prompt

# Generate improved summary
def generate_improved_summary(client, document_content: str, original_summary: str, 
                              eval_results: EvaluationResults, title: str) -> str:
    """Generate improved summary based on evaluation."""
    
    improvement_prompt = create_improvement_prompt(
        document_content,
        original_summary,
        eval_results,
        title
    )
    
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": improvement_prompt}
        ],
        response_format=DocumentAnalysis
    )
    
    # Fix: use correct response structure
    return response.choices[0].message.parsed.summary

# Run self-correction loop
print("=" * 80)
print("ORIGINAL EVALUATION RESULTS")
print("=" * 80)
print(f"Summarization Score: {eval_results.summarization_score:.4f}")
print(f"Coherence Score: {eval_results.coherence_score:.4f}")
print(f"Tonality Score: {eval_results.tonality_score:.4f}")
print(f"Safety Score: {eval_results.safety_score:.4f}")
print()

# Generate improved summary
print("Generating improved summary...")
improved_summary = generate_improved_summary(
    client,
    document_text,
    result.summary,
    eval_results,
    result.title
)

# Evaluate improved summary using same function evaluate_summary defined in previous section
print("Evaluating improved summary...")
improved_eval_results = evaluate_summary(document_text, improved_summary)

print()
print("=" * 80)
print("IMPROVED EVALUATION RESULTS")
print("=" * 80)
print(f"Summarization Score: {improved_eval_results.summarization_score:.4f}")
print(f"Coherence Score: {improved_eval_results.coherence_score:.4f}")
print(f"Tonality Score: {improved_eval_results.tonality_score:.4f}")
print(f"Safety Score: {improved_eval_results.safety_score:.4f}")
print()

# Comparison
print("=" * 80)
print("COMPARISON: ORIGINAL vs IMPROVED")
print("=" * 80)
improvements = {
    "Summarization": (eval_results.summarization_score, improved_eval_results.summarization_score),
    "Coherence": (eval_results.coherence_score, improved_eval_results.coherence_score),
    "Tonality": (eval_results.tonality_score, improved_eval_results.tonality_score),
    "Safety": (eval_results.safety_score, improved_eval_results.safety_score)
}

for metric, (original, improved) in improvements.items():
    change = improved - original
    pct_change = (change / original * 100) if original != 0 else 0
    status = "📈" if change > 0 else "📉" if change < 0 else "➡️"
    print(f"{status} {metric}: {original:.4f} → {improved:.4f} ({change:+.4f}, {pct_change:+.1f}%)")

print()
print("=" * 80)
print("ANALYSIS")
print("=" * 80)

avg_original = sum(s for s, _ in improvements.values()) / len(improvements)
avg_improved = sum(s for _, s in improvements.values()) / len(improvements)

print(f"Average Original Score: {avg_original:.4f}")
print(f"Average Improved Score: {avg_improved:.4f}")
print(f"Overall Improvement: {avg_improved - avg_original:+.4f}")

print(f"Summarization Reason:\n{improved_eval_results.summarization_reason}\n")
print(f"Coherence Reason:\n{improved_eval_results.coherence_reason}\n")
print(f"Tonality Reason:\n{improved_eval_results.tonality_reason}\n")
print(f"Safety Reason:\n{improved_eval_results.safety_reason}\n")

print()
print("IMPROVED SUMMARY:")
print("-" * 80)
print(improved_summary)

ORIGINAL EVALUATION RESULTS
Summarization Score: 0.7335
Coherence Score: 0.6566
Tonality Score: 0.6436
Safety Score: 0.6644

Generating improved summary...


✨ You're running DeepEval's latest Summarization [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

Evaluating improved summary...


✓ Evaluation completed 🎉! (time taken: 4.39s | token cost: 0.0070683 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


IMPROVED EVALUATION RESULTS
Summarization Score: 0.4856
Coherence Score: 0.3686
Tonality Score: 0.5892
Safety Score: 0.4188

COMPARISON: ORIGINAL vs IMPROVED
📉 Summarization: 0.7335 → 0.4856 (-0.2480, -33.8%)
📉 Coherence: 0.6566 → 0.3686 (-0.2880, -43.9%)
📉 Tonality: 0.6436 → 0.5892 (-0.0545, -8.5%)
📉 Safety: 0.6644 → 0.4188 (-0.2457, -37.0%)

ANALYSIS
Average Original Score: 0.6746
Average Improved Score: 0.4655
Overall Improvement: -0.2090
Summarization Reason:
The response captures the main theme of the GenAI Divide and mentions high adoption rates and low ROI, reflecting the key statistic of 95% failure. However, it lacks a detailed explanation of why AI initiatives fail to generate ROI and does not discuss the role of partnerships and external vendors effectively. Additionally, it doesn't explicitly identify methodology as the core issue behind the divide, limiting its alignment with the evaluation criteria.

Coherence Reason:
While the response attempts to capture the essence of


Did you get a better output?
- According to the Overall Improvement metric, the change in output summary was marginal despite calling the LLM to generate an improved output. One of the main benefits of this exercise is to show that the loop demonstrates iterative improvement capability

Why?
- The marginal change in improvement demonstrates how well the developer prompt and user prompt was defined. This gave the LLM concrete guidance to work within and generate a consistent output each time
- Further, these metrics rely on LLM's judgment which may be subjective or inconsistent even with low temperature or provided with previous summary to improve upon.

Are these enough?
- These evaluation steps being sufficient for this assignment, advanced cases could include fact-checking metric to verify claims against original, like a unit test or add A/B testing with multiple improved versions
- We can also Implement confidence scoring from the LLM judge and create domain-specific models for business/AI context


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
